# Objective

Validate whether the signal–horizon selection and weighting logic from Notebook 1 holds up out of sample using rolling train/test windows.

For each rolling training window, I:
1. score candidate signals across multiple forward-return horizons,
2. choose the best horizon for each selected signal,
3. compute signal weights using training data only,
4. construct a horizon-aware alpha signal,
5. apply that mapping to the next unseen test window.

This notebook is designed to answer a stricter question than Notebook 1:

**Do the selected signal–horizon pairs and weights still work when estimated only on past data and evaluated on future data?**

## 1. Import path and system setups

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

current = Path.cwd()
for p in [current, *current.parents]:
    if (p / "src").exists():
        project_root = p
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)

from src.scoring import (make_forward_returns,
                         score_signal_library,
                         summarize_signal_names)
from src.backtest import backtest_continuous_strategy
from src.utils import generate_rolling_windows, build_context_rank

Project root: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model


## 2. Load data

In [2]:
clean_close_prices = pd.read_csv(project_root / "data" / "processed" / "Phase1_Signal Discovery" / "notebook_02_data_prep" / "clean_close_prices.csv", index_col=0, parse_dates=True)
alpha_library_full = pd.read_parquet(project_root / "data" / "processed" / "Phase2_Signal Expansion" / "notebook_01_Multi-Horizon Signal Extraction" / "alpha_library_full.parquet")
context_library_full = pd.read_parquet(project_root / "data" / "processed" / "Phase2_Signal Expansion" / "notebook_01_Multi-Horizon Signal Extraction" / "context_library_full.parquet")
config_dir = project_root / "config"
with open(config_dir / "context_signals.json", "r") as f:
    CONTEXT_SIGNALS = json.load(f)

clean_close_prices = clean_close_prices.sort_index()
alpha_library_full = alpha_library_full.sort_index()
context_library_full = context_library_full.sort_index()

print("clean_close_prices:", clean_close_prices.shape)
print("alpha_library_full:", alpha_library_full.shape)
print("context_library_full:", context_library_full.shape)
print("Context signals:", CONTEXT_SIGNALS)

clean_close_prices: (2080, 31)
alpha_library_full: (2080, 90)
context_library_full: (2080, 180)
Context signals: ['volavg_60', 'mkt_regime_vol_60', 'volavg_20', 'mkt_regime_vol_20', 'vol_60', 'vol_20']


## 3. Walk-forward configuration

In [3]:
HORIZONS = {"fwd_1d": 1, "fwd_5d": 5, "fwd_10d": 10, "fwd_20d": 20}
TOP_N_SIGNALS = 3

TRAIN_SIZE = 252*2
TEST_SIZE = 252// 2
STEP_SIZE = 252// 2
PURGE_SIZE = 1
EMBARGO_SIZE = 0

## 4. Walk-forward validation

In [7]:
forward_returns = {
    horizon_name: make_forward_returns(clean_close_prices, horizon=horizon_days)
    for horizon_name, horizon_days in HORIZONS.items()}


def select_best_signal_horizon_pairs(
    alpha_library_train: pd.DataFrame,
    forward_returns_train: dict[str, pd.DataFrame],
    top_n: int = TOP_N_SIGNALS,
    min_obs: int = 60,
) -> pd.DataFrame:
    comparison_rows = []

    for horizon_name, fwd_ret_train in forward_returns_train.items():
        scores = score_signal_library(alpha_library_train, fwd_ret_train, min_obs=min_obs)
        summary = summarize_signal_names(scores, min_obs=min_obs).copy()
        summary["horizon"] = horizon_name
        comparison_rows.append(summary)

    signal_horizon_comparison = pd.concat(comparison_rows, ignore_index=True)

    best_pairs = (
        signal_horizon_comparison
        .sort_values(["signal", "mean_ic", "mean_spread"], ascending=[True, False, False])
        .groupby("signal", as_index=False)
        .first()
        .sort_values(["mean_ic", "mean_spread"], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )

    positive_mask = best_pairs["mean_ic"] > 0
    if positive_mask.any():
        best_pairs = best_pairs.loc[positive_mask].copy()
    if best_pairs.empty:
        raise ValueError("No signals selected in this training window.")

    horizon_days = best_pairs["horizon"].astype(str).str.extract(r"fwd_(\d+)d")[0]
    bad_mask = horizon_days.isna()
    if bad_mask.any():
        print("Could not parse horizon from these rows:")
        display(best_pairs.loc[bad_mask, ["signal", "horizon", "mean_ic", "mean_spread"]])
        raise ValueError("Found invalid horizon values in best_pairs. Expected strings like 'fwd_5d'.")

    best_pairs["horizon_days"] = horizon_days.astype(int)
    best_pairs["raw_weight"] = best_pairs["mean_ic"].clip(lower=0)

    if best_pairs["raw_weight"].sum() == 0:
        best_pairs["raw_weight"] = 1.0 / len(best_pairs)

    best_pairs["weight"] = best_pairs["raw_weight"] / best_pairs["raw_weight"].sum()

    return best_pairs[
        ["signal", "horizon", "horizon_days", "mean_ic", "mean_spread", "raw_weight", "weight"]
    ]


def build_alpha_rank_from_selected_signals(
    alpha_library: pd.DataFrame,
    selected_pairs: pd.DataFrame,
) -> pd.DataFrame:
    if not isinstance(alpha_library.columns, pd.MultiIndex):
        raise ValueError("alpha_library must have MultiIndex columns with a 'signal' level.")
    if "signal" not in alpha_library.columns.names:
        raise ValueError(
            f"alpha_library columns must include a 'signal' level. Found: {alpha_library.columns.names}"
        )

    alpha_components = []

    for _, row in selected_pairs.iterrows():
        signal_rank = alpha_library.xs(row["signal"], level="signal", axis=1).rank(axis=1, pct=True)
        alpha_components.append(signal_rank * row["weight"])

    alpha_score = sum(alpha_components)
    return alpha_score.rank(axis=1, pct=True)


def build_weighted_horizon_aware_portfolio_target(
    forward_returns_map: dict[str, pd.DataFrame],
    selected_pairs: pd.DataFrame,
    columns: pd.Index | None = None,
) -> pd.DataFrame:
    portfolio_target = None

    for _, row in selected_pairs.iterrows():
        weighted_target = forward_returns_map[row["horizon"]] * row["weight"]
        if portfolio_target is None:
            portfolio_target = weighted_target.copy()
        else:
            portfolio_target = portfolio_target.add(weighted_target, fill_value=0.0)

    if columns is not None:
        portfolio_target = portfolio_target.reindex(columns=columns)

    return portfolio_target


rolling_windows = generate_rolling_windows(
    index=clean_close_prices.index,
    train_size=int(TRAIN_SIZE),
    test_size=int(TEST_SIZE),
    step_size=int(STEP_SIZE),
    purge_size=PURGE_SIZE,
    embargo_size=EMBARGO_SIZE,
)

window_results = []
selected_pair_logs = []
window_returns = []

for window_id, (train_idx, test_idx) in enumerate(rolling_windows, start=1):
    alpha_library_train = alpha_library_full.loc[train_idx]
    alpha_library_test = alpha_library_full.loc[test_idx]
    context_library_test = context_library_full.loc[test_idx]

    forward_returns_train = {
        horizon_name: fwd_ret.loc[train_idx]
        for horizon_name, fwd_ret in forward_returns.items()
    }
    forward_returns_test = {
        horizon_name: fwd_ret.loc[test_idx]
        for horizon_name, fwd_ret in forward_returns.items()
    }

    selected_pairs = select_best_signal_horizon_pairs(
        alpha_library_train=alpha_library_train,
        forward_returns_train=forward_returns_train,
        top_n=TOP_N_SIGNALS,
        min_obs=60,
    )

    alpha_rank_test = build_alpha_rank_from_selected_signals(
        alpha_library=alpha_library_test,
        selected_pairs=selected_pairs,
    )

    context_rank_test = build_context_rank(
        context_library=context_library_test,
        context_signals=CONTEXT_SIGNALS,
    ).reindex(index=alpha_rank_test.index, columns=alpha_rank_test.columns)

    portfolio_target_test = build_weighted_horizon_aware_portfolio_target(
        forward_returns_map=forward_returns_test,
        selected_pairs=selected_pairs,
        columns=alpha_rank_test.columns,
    ).reindex(index=alpha_rank_test.index, columns=alpha_rank_test.columns)

    wf_result = backtest_continuous_strategy(
        alpha_rank=alpha_rank_test,
        forward_returns=portfolio_target_test,
        context_rank=context_rank_test,
    )

    window_results.append({
        "window_id": window_id,
        "train_start": train_idx.min(),
        "train_end": train_idx.max(),
        "test_start": test_idx.min(),
        "test_end": test_idx.max(),
        "n_selected_pairs": len(selected_pairs),
        "sharpe": wf_result["sharpe"],
        "max_drawdown": wf_result["max_drawdown"],
        "total_return": wf_result["total_return"],
        "mean_turnover": wf_result["turnover"].mean(),
        "mean_cost": wf_result["costs"].mean(),
    })

    selected_pair_logs.append(
        selected_pairs.assign(
            window_id=window_id,
            train_start=train_idx.min(),
            train_end=train_idx.max(),
            test_start=test_idx.min(),
            test_end=test_idx.max(),
        )
    )

    window_returns.append(wf_result["net_returns"].rename(f"window_{window_id}"))

walk_forward_results = pd.DataFrame(window_results)
selected_signal_horizon_log = pd.concat(selected_pair_logs, ignore_index=True)
walk_forward_return_panel = pd.concat(window_returns, axis=1)

display(walk_forward_results)
display(selected_signal_horizon_log.head(20))


,window_id,train_start,train_end,test_start,test_end,n_selected_pairs,sharpe,max_drawdown,total_return,mean_turnover,mean_cost
0,1,2018-01-02,2020-01-02,2020-01-06,2020-07-06,3,0.296320,-0.066334,0.009343,0.064896,0.000032
1,2,2018-07-03,2020-07-02,2020-07-07,2021-01-04,3,0.191435,-0.069371,0.005432,0.206955,0.000103
2,3,2019-01-03,2020-12-31,2021-01-05,2021-07-06,3,0.832952,-0.054811,0.026065,0.190742,0.000095
3,4,2019-07-05,2021-07-02,2021-07-07,2022-01-03,3,-0.922047,-0.123011,-0.038448,0.273690,0.000137
4,5,2020-01-03,2021-12-31,2022-01-04,2022-07-06,3,-0.002917,-0.104479,-0.001211,0.123211,0.000062
5,6,2020-07-06,2022-07-05,2022-07-07,2023-01-04,3,0.223373,-0.082639,0.007494,0.098514,0.000049
6,7,2021-01-04,2023-01-03,2023-01-05,2023-07-07,3,-0.883241,-0.179282,-0.049264,0.099588,0.000050
7,8,2021-07-06,2023-07-06,2023-07-10,2024-01-05,3,-1.470378,-0.162695,-0.063602,0.080847,0.000040
8,9,2022-01-03,2024-01-04,2024-01-08,2024-07-09,3,1.010509,-0.140281,0.047112,0.114417,0.000057
9,10,2022-07-06,2024-07-08,2024-07-10,2025-01-07,3,-0.523184,-0.140749,-0.019719,0.066975,0.000033


,signal,horizon,horizon_days,mean_ic,mean_spread,raw_weight,weight,window_id,train_start,train_end,test_start,test_end
0,rev_5,fwd_10d,10,0.097187,0.010043,0.097187,0.367480,1,2018-01-02,2020-01-02,2020-01-06,2020-07-06
1,breakout_dn_20,fwd_20d,20,0.083711,0.001154,0.083711,0.316526,1,2018-01-02,2020-01-02,2020-01-06,2020-07-06
2,breakout_dn_60,fwd_10d,10,0.083571,0.000634,0.083571,0.315994,1,2018-01-02,2020-01-02,2020-01-06,2020-07-06
3,breakout_dn_60,fwd_1d,1,0.097884,0.000384,0.097884,0.449755,2,2018-07-03,2020-07-02,2020-07-07,2021-01-04
4,breakout_dn_20,fwd_1d,1,0.068352,0.000386,0.068352,0.314063,2,2018-07-03,2020-07-02,2020-07-07,2021-01-04
5,rev_5,fwd_20d,20,0.051402,0.006089,0.051402,0.236182,2,2018-07-03,2020-07-02,2020-07-07,2021-01-04
6,breakout_dn_60,fwd_1d,1,0.121303,0.000363,0.121303,0.467068,3,2019-01-03,2020-12-31,2021-01-05,2021-07-06
7,breakout_dn_20,fwd_1d,1,0.077030,0.000406,0.077030,0.296598,3,2019-01-03,2020-12-31,2021-01-05,2021-07-06
8,rev_5,fwd_20d,20,0.061379,0.008944,0.061379,0.236334,3,2019-01-03,2020-12-31,2021-01-05,2021-07-06
9,breakout_dn_60,fwd_1d,1,0.112812,0.000376,0.112812,0.425640,4,2019-07-05,2021-07-02,2021-07-07,2022-01-03


# Midpoint Conclusion 1:

The walk-forward results confirm that the signal framework captures real predictive edge, but the strategy is not yet stable or robust.

Performance varies widely across windows, with strong gains in some periods and significant losses in others. This indicates the signals are regime-dependent, working well under certain conditions but failing under others.

The primary issue is the instability of signal-horizon selection. Optimal horizons shift frequently across windows, leading to inconsistent signal composition, volatile weights, and uneven portfolio behavior.

In summary:

- The signal discovery process is valid (edge exists)
- The current strategy is unstable (poor generalization)
- The alpha is conditional, not persistent

The next step is to stabilize the model by enforcing more consistent horizon usage, filtering persistent signals, and reducing variability in portfolio construction.

## 5. Signal stabilization

In [8]:
num_windows = selected_signal_horizon_log["window_id"].nunique()
MIN_WINDOWS_PERSISTENT = max(2, int(np.ceil(num_windows / 2)))
MIN_HORIZON_SHARE = 0.6

signal_frequency = (
    selected_signal_horizon_log
    .groupby("signal", as_index=False)
    .agg(
        n_windows=("window_id", "nunique"),
        mean_ic=("mean_ic", "mean"),
        mean_weight=("weight", "mean"),
    )
    .sort_values(["n_windows", "mean_ic"], ascending=[False, False])
    .reset_index(drop=True)
)

signal_frequency["persistence_ratio"] = signal_frequency["n_windows"] / num_windows
signal_frequency["is_persistent"] = signal_frequency["n_windows"] >= MIN_WINDOWS_PERSISTENT

horizon_frequency_by_signal = (
    selected_signal_horizon_log
    .groupby(["signal", "horizon"], as_index=False)
    .agg(
        n_windows=("window_id", "nunique"),
        mean_ic=("mean_ic", "mean"),
        mean_weight=("weight", "mean"),
    )
    .sort_values(["signal", "n_windows", "mean_ic"], ascending=[True, False, False])
    .reset_index(drop=True)
)

signal_horizon_stability = (
    horizon_frequency_by_signal
    .groupby("signal", as_index=False)
    .agg(
        n_horizons=("horizon", "nunique"),
        chosen_horizon=("horizon", "first"),
        chosen_horizon_windows=("n_windows", "first"),
        chosen_horizon_mean_ic=("mean_ic", "first"),
        chosen_horizon_mean_weight=("mean_weight", "first"),
    )
)

signal_horizon_stability = signal_horizon_stability.merge(
    signal_frequency[["signal", "n_windows", "persistence_ratio", "is_persistent"]],
    on="signal",
    how="left",
)

signal_horizon_stability["chosen_horizon_ratio"] = (
    signal_horizon_stability["chosen_horizon_windows"] / signal_horizon_stability["n_windows"]
)
signal_horizon_stability["horizon_stability"] = np.where(
    signal_horizon_stability["chosen_horizon_ratio"] >= MIN_HORIZON_SHARE,
    "stable",
    "unstable",
)

signal_stability_summary = signal_horizon_stability.sort_values(
    ["is_persistent", "chosen_horizon_ratio", "n_windows", "chosen_horizon_mean_ic"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

stabilized_signal_horizon_table = (
    signal_stability_summary.loc[signal_stability_summary["is_persistent"]].copy()
)

if stabilized_signal_horizon_table.empty:
    raise ValueError("stabilized_signal_horizon_table is empty. No signals passed the persistence filter.")

horizon_days = (
    stabilized_signal_horizon_table["chosen_horizon"]
    .astype(str)
    .str.extract(r"fwd_(\d+)d")[0]
)

bad_mask = horizon_days.isna()
if bad_mask.any():
    print("Could not parse chosen_horizon for these rows:")
    display(
        stabilized_signal_horizon_table.loc[
            bad_mask,
            ["signal", "chosen_horizon", "chosen_horizon_windows", "chosen_horizon_ratio"]
        ]
    )
    raise ValueError("Invalid chosen_horizon values found. Expected strings like 'fwd_5d'.")

stabilized_signal_horizon_table["horizon_days"] = horizon_days.astype(int)

stabilized_signal_horizon_table["raw_weight"] = (
    stabilized_signal_horizon_table["chosen_horizon_mean_weight"]
)

raw_sum = stabilized_signal_horizon_table["raw_weight"].sum()
if raw_sum == 0:
    stabilized_signal_horizon_table["raw_weight"] = 1.0 / len(stabilized_signal_horizon_table)
    raw_sum = stabilized_signal_horizon_table["raw_weight"].sum()

stabilized_signal_horizon_table["weight"] = stabilized_signal_horizon_table["raw_weight"] / raw_sum

stabilized_signal_horizon_table = stabilized_signal_horizon_table[
    [
        "signal",
        "n_windows",
        "persistence_ratio",
        "chosen_horizon",
        "horizon_days",
        "chosen_horizon_windows",
        "chosen_horizon_ratio",
        "horizon_stability",
        "chosen_horizon_mean_ic",
        "raw_weight",
        "weight",
    ]
].sort_values(
    ["weight", "chosen_horizon_ratio", "n_windows"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(pd.DataFrame({
    "num_windows": [num_windows],
    "min_windows_persistent": [MIN_WINDOWS_PERSISTENT],
    "min_horizon_share": [MIN_HORIZON_SHARE],
}))

display(signal_frequency)
display(horizon_frequency_by_signal)
display(signal_stability_summary)
display(stabilized_signal_horizon_table)


,num_windows,min_windows_persistent,min_horizon_share
0,12,6,0.6


,signal,n_windows,mean_ic,mean_weight,persistence_ratio,is_persistent
0,breakout_dn_60,12,0.085515,0.439331,1.0,True
1,breakout_dn_20,12,0.056763,0.278048,1.0,True
2,rev_5,12,0.056165,0.282621,1.0,True


,signal,horizon,n_windows,mean_ic,mean_weight
0,breakout_dn_20,fwd_1d,4,0.076682,0.308095
1,breakout_dn_20,fwd_20d,4,0.050103,0.267115
2,breakout_dn_20,fwd_10d,4,0.043505,0.258936
3,breakout_dn_60,fwd_20d,5,0.075846,0.495646
4,breakout_dn_60,fwd_1d,4,0.110494,0.443954
5,breakout_dn_60,fwd_10d,3,0.068323,0.339308
6,rev_5,fwd_20d,8,0.057632,0.272079
7,rev_5,fwd_10d,2,0.074386,0.360184
8,rev_5,fwd_5d,1,0.033120,0.247572
9,rev_5,fwd_1d,1,0.031027,0.246874


,signal,n_horizons,chosen_horizon,chosen_horizon_windows,chosen_horizon_mean_ic,chosen_horizon_mean_weight,n_windows,persistence_ratio,is_persistent,chosen_horizon_ratio,horizon_stability
0,rev_5,4,fwd_20d,8,0.057632,0.272079,12,1.0,True,0.666667,stable
1,breakout_dn_60,3,fwd_20d,5,0.075846,0.495646,12,1.0,True,0.416667,unstable
2,breakout_dn_20,3,fwd_1d,4,0.076682,0.308095,12,1.0,True,0.333333,unstable


,signal,n_windows,persistence_ratio,chosen_horizon,horizon_days,chosen_horizon_windows,chosen_horizon_ratio,horizon_stability,chosen_horizon_mean_ic,raw_weight,weight
0,breakout_dn_60,12,1.0,fwd_20d,20,5,0.416667,unstable,0.075846,0.495646,0.460715
1,breakout_dn_20,12,1.0,fwd_1d,1,4,0.333333,unstable,0.076682,0.308095,0.286381
2,rev_5,12,1.0,fwd_20d,20,8,0.666667,stable,0.057632,0.272079,0.252904


# Midpoint Conclusion 2:

The stabilization analysis shows that signal selection is highly persistent across walk-forward windows, with all three core signals recurring in every window. 

However, horizon selection is less stable: only rev_5 maintains a consistent dominant horizon, while the breakout signals shift across forecast horizons over time. This suggests that the underlying signals are robust, but their optimal holding periods remain regime-dependent. 

The next step is to convert this stable signal set into a more robust portfolio rule by reducing horizon instability and testing whether a simplified, fixed mapping improves out-of-sample consistency.

## 6. Fixed-horizon signal validation